# Pairfocus Horn-Rule Extraction — Smoke Test

This notebook runs a small end-to-end test of the pairfocus experiment:
**5 learner iterations, 1 run, 1 link pair, for each of the four models**
(LINREG, MLP/FF, GN, GCNII).

It drives `src/run_network_pairfocus.py`, which applies the sampling HORN
algorithm to a trained traffic model treated as an oracle, and writes the
extracted Horn rules to `results/`.

The full published experiment uses the same runner with all 35 link pairs, both
orientations, 5 runs, and a 500-iteration cap (see `README.md`).

## 1. Environment

Create and activate the conda environment before launching Jupyter:

```bash
conda env create -f environment.yml
conda activate horn-pairfocus-repro
```

Everything runs on CPU. Each model is launched as its own subprocess and loads
its checkpoint plus the small static-input file, so the four runs are quick.
The full ~2 GB MATSim training dataset is not needed and is not included.

## 2. Configuration

In [ ]:
import os
import sys
from pathlib import Path

# This notebook lives at the package root.
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

# --- Experiment configuration (smoke test) ---
MODELS = ["LINREG", "MLP", "GN", "GCNII"]
PAIR = 33             # focus link pair (see data/link_pairs.csv)
ORIENTATION = "both"  # "A", "B", or "both"
ITERATIONS = 5        # learner iteration cap (full experiment uses 500)
RUNS = 1              # runs per pair/orientation (full experiment uses 5)
N_EQ_SAMPLES = 500        # random final equivalence-query samples
N_POSITIVE_PER_EQ = 50    # seeded positive samples before the random phase
THRESHOLD = 5.0           # congestion speed threshold in m/s
HOPS = 1                  # neighbourhood hop radius
SEED = 1

# Use the corrected physical-adjacency neighbourhood table (paper-final).
NEIGHBOURHOOD_CSV = DATA_DIR / "link_pair_neighbourhood_physical_candidate.csv"
os.environ["PAIRFOCUS_NEIGHBOURHOOD_CSV"] = str(NEIGHBOURHOOD_CSV)

print("Project root      :", PROJECT_ROOT)
print("Models            :", MODELS)
print(f"pair={PAIR}  orientation={ORIENTATION}  iterations={ITERATIONS}  runs={RUNS}")
print("Neighbourhood CSV :", NEIGHBOURHOOD_CSV.name)

In [ ]:
# Verify all required inputs are present before running.
required = [
    SRC_DIR / "run_network_pairfocus.py",
    DATA_DIR / "pairfocus_static_inputs.pkl",
    DATA_DIR / "exttest_capacity_scaler.pkl",
    DATA_DIR / "exttest_speed_scaler.pkl",
    NEIGHBOURHOOD_CSV,
    MODELS_DIR / "used_hparams.json",
]
required += [MODELS_DIR / f"model_state_{m.lower()}.pt" for m in MODELS]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))
print("All required files present.")

## 3. Run the experiment

Each model is executed in a fresh Python process (the runner loads its model at
import time). Progress bars and per-pair summaries are printed below.

In [ ]:
import subprocess
import time

RESULTS_DIR.mkdir(exist_ok=True)

def run_model(model_key):
    cmd = [
        sys.executable, "run_network_pairfocus.py",
        "--model", model_key,
        "--pairs", str(PAIR),
        "--orientation", ORIENTATION,
        "--hops", str(HOPS),
        "--threshold", str(THRESHOLD),
        "--iterations", str(ITERATIONS),
        "--n-eq-samples", str(N_EQ_SAMPLES),
        "--n-positive-per-eq", str(N_POSITIVE_PER_EQ),
        "--runs", str(RUNS),
        "--seed", str(SEED),
    ]
    print(f"\n=== {model_key} ===")
    print(" ".join(cmd))
    t0 = time.time()
    # Run from src/ so imports resolve; the runner writes to <package>/results
    # regardless of cwd (its output path is computed from __file__).
    proc = subprocess.run(cmd, cwd=SRC_DIR, capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        print("STDERR (tail):\n", proc.stderr[-2000:])
        raise RuntimeError(f"{model_key} run failed (exit {proc.returncode})")
    print(f"{model_key} done in {time.time() - t0:.1f}s")

for m in MODELS:
    run_model(m)

## 4. Inspect extracted rules

For each model, show the cleaned Horn rules for the focus pair and orientation(s).

In [ ]:
import csv

def latest_result_dir(model_key):
    pattern = f"horn_network_pairfocus_*_{model_key}_without_neigh_seed{SEED}_*"
    dirs = sorted(RESULTS_DIR.glob(pattern), key=lambda p: p.stat().st_mtime)
    return dirs[-1] if dirs else None

for m in MODELS:
    d = latest_result_dir(m)
    print(f"\n########## {m}: {d.name if d else 'NO RESULT DIR'} ##########")
    if d is None:
        continue
    clean_dir = d / "output_data" / "hornRulesClean"
    for csv_path in sorted(clean_dir.glob("*.csv")):
        print(f"\n--- {csv_path.name} ---")
        with open(csv_path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        for row in rows:
            print("   ", row[0] if row else "")